In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import holidays
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [2]:
df = pd.read_csv('dados/station_data_dataverse.csv')
df.columns = df.columns.str.lower()
df = df.drop_duplicates()

In [3]:
df['created'] =pd.to_datetime(df['created'], errors='coerce')
df = df.dropna(subset=['created', 'kwhtotal'])
df= df[df['kwhtotal'] > 0]
df.shape

(3340, 24)

In [4]:
serie = df.set_index('created')['kwhtotal'].resample('D').sum()
serie = serie.asfreq('D', fill_value=0)
serie.shape

(321,)

In [5]:
print(serie.index.min(), serie.index.max())
print((serie == 0).sum(), 'dias sem sessão')

dias_zero = serie[serie == 0]
print(dias_zero.index.dayofweek.value_counts().sort_index())

0014-11-18 00:00:00 0015-10-04 00:00:00
84 dias sem sessão
created
0     8
1     5
2     4
3     6
4     9
5    20
6    32
Name: count, dtype: int64


In [7]:
feriados = holidays.US()

dados = pd.DataFrame({'y': serie})
dados['dia_semana'] = dados.index.dayofweek
dados['mes'] = dados.index.month
dados['feriado'] = [int(d in feriados) for d in dados.index]
dados.head()

,y,dia_semana,mes,feriado
created,,,,
14-11-18,13.39,1,11,0
14-11-19,18.77,2,11,0
14-11-20,6.95,3,11,0
14-11-21,14.14,4,11,0
14-11-22,0.00,5,11,0


In [8]:
corte = int(len(dados) * 0.8)
treino, teste = dados.iloc[:corte], dados.iloc[corte:]
len(treino), len(teste)

(256, 65)

In [ ]:
def avaliar(real, previsto):
    mae = mean_absolute_error(real, previsto)
    rmse = np.sqrt(mean_squared_error(real, previsto))
    return round(mae, 2), round(rmse, 2)

resultados, previsoes = {}, {}

previsoes['baseline'] = np.full(len(teste), treino['y'].mean())
resultados['baseline'] = avaliar(teste['y'], previsoes['baseline'])